# D. Backbone Experiment Matrix (No MMDet)

`05B` smoke 통과 후 **본실험**용.

권장 순서:
1. `resnet50_fpn` × `original` / `clahe_gamma` (EPOCHS 올리기)
2. `mobilenet_v3_large_fpn` × `clahe_gamma`
3. (여유) YOLO11s on best pipeline
4. 상위 조합만 남겨 ConvNeXt/Swin 커스텀은 후순위

결과: `ARCADE/tv_backbone_exps/`

In [ ]:
from pathlib import Path
import json, time, random
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    fasterrcnn_resnet50_fpn_v2,
    fasterrcnn_mobilenet_v3_large_fpn,
    FasterRCNN_ResNet50_FPN_Weights,
    FasterRCNN_ResNet50_FPN_V2_Weights,
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

ARCADE = Path.cwd().parent
DATA = ARCADE / 'data'
STEN = DATA / 'arcade' / 'arcade' / 'stenosis'
PREP = DATA / 'preprocessed'
YOLO = DATA / 'yolo'
WORK = ARCADE / 'tv_backbone_exps'
WORK.mkdir(parents=True, exist_ok=True)

PIPE_IMG = {
    'original': {
        'train': STEN / 'train' / 'images',
        'val': STEN / 'val' / 'images',
    },
    'clahe': {
        'train': PREP / '01_clahe' / 'train',
        'val': PREP / '01_clahe' / 'val',
    },
    'clahe_gamma': {
        'train': PREP / '02_clahe_gamma' / 'train',
        'val': PREP / '02_clahe_gamma' / 'val',
    },
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('ARCADE', ARCADE)
print('device', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')
print('WORK', WORK)

## Hyperparams

Smoke는 5 epoch로 충분. 본실험은 **12~24**부터. loss가 계속 떨어지고 AP50이 오르면 더 올려도 됨.
과적합 신호(train loss↓ / val AP↓)면 early stop 또는 LR 낮추기.

In [1]:
# === 본실험 기본값 (smoke=5 통과 후) ===
EPOCHS = 20
BATCH = 4
LR = 5e-4
NUM_WORKERS = 0  # Jupyter 안정
PRINT_EVERY = 50
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

# 권장 순서 매트릭스 — 위에서부터 돌리기
EXPS = [
    # phase 1: ResNet50 vs preprocess
    ('resnet50_fpn', 'original'),
    ('resnet50_fpn', 'clahe_gamma'),
    # phase 2: lighter backbone
    ('mobilenet_v3_large_fpn', 'clahe_gamma'),
    # optional: ResNet v2
    # ('resnet50_fpn_v2', 'clahe_gamma'),
]

# True면 아래 전체 EXPS 순차 실행 / False면 한 개만
RUN_ALL = False
SINGLE = ('resnet50_fpn', 'clahe_gamma')  # RUN_ALL=False 일 때

print('EPOCHS', EPOCHS, 'BATCH', BATCH, 'RUN_ALL', RUN_ALL)

NameError: name 'random' is not defined

In [ ]:
def load_stenosis_records(split: str, img_root: Path):
    ann = json.loads((STEN / split / 'annotations' / f'{split}.json').read_text())
    id2file = {im['id']: im['file_name'] for im in ann['images']}
    id2wh = {im['id']: (im['width'], im['height']) for im in ann['images']}
    by = defaultdict(list)
    for a in ann['annotations']:
        if a.get('category_id') != 26:
            continue
        x, y, w, h = a['bbox']
        if w <= 1 or h <= 1:
            continue
        by[a['image_id']].append([x, y, x + w, y + h])
    records = []
    missing = 0
    for iid, boxes in by.items():
        p = img_root / id2file[iid]
        if not p.exists():
            missing += 1
            continue
        records.append({'path': p, 'boxes': boxes, 'wh': id2wh[iid]})
    return records, missing


class StenosisDetDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        r = self.records[i]
        img = read_image(str(r['path'])).float() / 255.0
        if img.shape[0] == 1:
            img = img.repeat(3, 1, 1)
        elif img.shape[0] == 4:
            img = img[:3]
        boxes = torch.tensor(r['boxes'], dtype=torch.float32)
        labels = torch.ones((boxes.shape[0],), dtype=torch.int64)  # stenosis=1
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([i]),
        }
        return img, target


def collate_fn(batch):
    return tuple(zip(*batch))


def make_loaders(pipe, batch_size=4, num_workers=0):
    roots = PIPE_IMG[pipe]
    tr, mtr = load_stenosis_records('train', roots['train'])
    va, mva = load_stenosis_records('val', roots['val'])
    print(f'[{pipe}] train={len(tr)} (miss {mtr})  val={len(va)} (miss {mva})')
    train_loader = DataLoader(StenosisDetDataset(tr), batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, collate_fn=collate_fn)
    val_loader = DataLoader(StenosisDetDataset(va), batch_size=1, shuffle=False,
                            num_workers=num_workers, collate_fn=collate_fn)
    return train_loader, val_loader


def build_model(name: str, num_classes=2):
    if name == 'resnet50_fpn':
        m = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    elif name == 'resnet50_fpn_v2':
        m = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
    elif name == 'mobilenet_v3_large_fpn':
        m = fasterrcnn_mobilenet_v3_large_fpn(weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT)
    else:
        raise ValueError(name)
    in_feat = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
    return m

In [ ]:
def box_iou(a, b):
    if a.numel() == 0 or b.numel() == 0:
        return torch.zeros((a.shape[0], b.shape[0]))
    lt = torch.max(a[:, None, :2], b[None, :, :2])
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    area_a = (a[:, 2] - a[:, 0]).clamp(min=0) * (a[:, 3] - a[:, 1]).clamp(min=0)
    area_b = (b[:, 2] - b[:, 0]).clamp(min=0) * (b[:, 3] - b[:, 1]).clamp(min=0)
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-6)


def ap50(pred_boxes, pred_scores, gt_boxes, iou_thr=0.5):
    """image-level contribution lists for VOC-style AP50 aggregation."""
    if len(pred_boxes) == 0:
        return [], int(len(gt_boxes))
    order = pred_scores.argsort(descending=True)
    pred_boxes = pred_boxes[order]
    pred_scores = pred_scores[order]
    matched = set()
    hits = []
    if len(gt_boxes):
        ious = box_iou(pred_boxes, gt_boxes)
    else:
        ious = None
    for i in range(len(pred_boxes)):
        score = float(pred_scores[i])
        if ious is None or ious.numel() == 0:
            hits.append((score, 0))
            continue
        j = int(ious[i].argmax())
        if float(ious[i, j]) >= iou_thr and j not in matched:
            matched.add(j)
            hits.append((score, 1))
        else:
            hits.append((score, 0))
    return hits, int(len(gt_boxes))


def evaluate_ap50(model, loader, score_thr=0.05):
    model.eval()
    all_hits = []
    n_gt = 0
    with torch.no_grad():
        for images, targets in loader:
            images = [im.to(device) for im in images]
            outs = model(images)
            for out, tgt in zip(outs, targets):
                keep = out['scores'] >= score_thr
                hits, g = ap50(out['boxes'][keep].cpu(), out['scores'][keep].cpu(), tgt['boxes'].cpu())
                all_hits.extend(hits)
                n_gt += g
    if n_gt == 0 or not all_hits:
        return 0.0
    all_hits.sort(key=lambda x: -x[0])
    tp = fp = 0
    prec, rec = [], []
    for _, h in all_hits:
        if h:
            tp += 1
        else:
            fp += 1
        prec.append(tp / (tp + fp))
        rec.append(tp / n_gt)
    # 11-point VOC AP
    ap = 0.0
    for t in [i / 10 for i in range(11)]:
        ps = [p for p, r in zip(prec, rec) if r >= t]
        ap += max(ps) if ps else 0.0
    return ap / 11


def train_one(backbone, pipe, epochs=EPOCHS, batch=BATCH, lr=LR):
    out = WORK / f'frcnn_{backbone}__{pipe}'
    out.mkdir(parents=True, exist_ok=True)
    train_loader, val_loader = make_loaders(pipe, batch_size=batch, num_workers=NUM_WORKERS)
    model = build_model(backbone).to(device)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    hist = []
    best_ap = -1.0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        loss_sum, n = 0.0, 0
        for step, (images, targets) in enumerate(train_loader, 1):
            images = [im.to(device) for im in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            loss_sum += float(loss.detach())
            n += 1
            if step % PRINT_EVERY == 0:
                print(f'  ep{ep} step{step} loss={loss_sum/n:.4f}')
        sched.step()
        ap = evaluate_ap50(model, val_loader)
        row = {'epoch': ep, 'train_loss': loss_sum / max(n, 1), 'ap50': ap, 'lr': sched.get_last_lr()[0]}
        hist.append(row)
        print(f'[{backbone}|{pipe}] ep{ep}/{epochs} loss={row["train_loss"]:.4f} AP50={ap:.4f}')
        torch.save({'model': model.state_dict(), 'epoch': ep, 'ap50': ap}, out / 'last.pt')
        if ap >= best_ap:
            best_ap = ap
            torch.save({'model': model.state_dict(), 'epoch': ep, 'ap50': ap}, out / 'best.pt')
    meta = {
        'backbone': backbone,
        'pipeline': pipe,
        'epochs': epochs,
        'batch': batch,
        'lr': lr,
        'best_ap50': best_ap,
        'seconds': time.time() - t0,
        'history': hist,
    }
    (out / 'metrics.json').write_text(json.dumps(meta, indent=2))
    print('saved', out, 'best_ap50', best_ap)
    return meta

## Run FRCNN matrix

먼저 `RUN_ALL=False` + `SINGLE=('resnet50_fpn','clahe_gamma')` + `EPOCHS=20` 한 방.
끝나면 `RUN_ALL=True`로 phase1~2 돌리기.

In [ ]:
todo = EXPS if RUN_ALL else [SINGLE]
results = []
for bb, pipe in todo:
    print('\n====', bb, pipe, '====')
    results.append(train_one(bb, pipe))

print('\n=== summary ===')
for r in results:
    print(f"{r['backbone']:28s} {r['pipeline']:12s} best_AP50={r['best_ap50']:.4f}  time={r['seconds']/60:.1f}m")

In [ ]:
# Aggregate all existing FRCNN runs under WORK
import pandas as pd

rows = []
for d in sorted(WORK.glob('frcnn_*')):
    mj = d / 'metrics.json'
    if not mj.exists():
        continue
    m = json.loads(mj.read_text())
    rows.append({
        'exp': d.name,
        'backbone': m.get('backbone'),
        'pipeline': m.get('pipeline'),
        'epochs': m.get('epochs'),
        'best_ap50': m.get('best_ap50'),
        'last_ap50': (m.get('history') or [{}])[-1].get('ap50'),
        'minutes': round(m.get('seconds', 0) / 60, 1),
    })
df = pd.DataFrame(rows).sort_values('best_ap50', ascending=False) if rows else pd.DataFrame()
display(df)
if len(df):
    df.to_csv(WORK / 'summary_frcnn.csv', index=False)
    print('wrote', WORK / 'summary_frcnn.csv')

## (여유) YOLO11 scale on best pipeline

FRCNN 상위 pipeline 고른 뒤 YOLO11s 비교. Ultralytics 필요.

In [ ]:
RUN_YOLO = False  # True로 켠 뒤 실행

BEST_PIPE = 'clahe_gamma'  # FRCNN에서 이긴 preprocess
YOLO_MODELS = ['yolo11n.pt', 'yolo11s.pt']  # 여유 있으면 m
YOLO_EPOCHS = 50
YOLO_IMGSZ = 640

if RUN_YOLO:
    from ultralytics import YOLO

    yroot = YOLO / BEST_PIPE
    data_yaml = WORK / f'yolo_{BEST_PIPE}.yaml'
    data_yaml.write_text(
        f"path: {yroot.as_posix()}\n"
        f"train: images/train\n"
        f"val: images/val\n"
        f"names:\n  0: stenosis\n"
    )
    yolo_rows = []
    for weights in YOLO_MODELS:
        name = f"yolo11_{weights.replace('.pt','')}__{BEST_PIPE}"
        print('====', name)
        model = YOLO(weights)
        res = model.train(
            data=str(data_yaml),
            epochs=YOLO_EPOCHS,
            imgsz=YOLO_IMGSZ,
            project=str(WORK / 'yolo_scales'),
            name=name,
            exist_ok=True,
            workers=0,
        )
        metrics = getattr(res, 'results_dict', None) or {}
        yolo_rows.append({'name': name, **{k: metrics.get(k) for k in metrics}})
        print(name, metrics)
    (WORK / 'summary_yolo_scales.json').write_text(json.dumps(yolo_rows, indent=2, default=str))
else:
    print('RUN_YOLO=False — FRCNN 끝난 뒤 True로')

## 후순위: ConvNeXt / Swin 커스텀 (스켈레톤만)

상위 preprocess + detector 확정 후에만. 지금은 실행하지 말 것.

In [ ]:
# Skeleton only — do not run until FRCNN/YOLO matrix is done.
RUN_CUSTOM_BACKBONE = False

if RUN_CUSTOM_BACKBONE:
    from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
    from torchvision.models.detection.backbone_utils import BackboneWithFPN
    from torchvision.models.detection import FasterRCNN
    from torchvision.ops.feature_pyramid_network import LastLevelMaxPool
    from torchvision.models.feature_extraction import create_feature_extractor

    def convnext_fpn_backbone():
        # ConvNeXt stage channels: rough mapping — verify before serious runs
        body = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT).features
        # return_layers must match actual module names; inspect once before train
        return_layers = {'1': '0', '3': '1', '5': '2', '7': '3'}
        feat = create_feature_extractor(body, return_nodes=return_layers)
        # channel sizes for convnext_tiny stages (approx)
        in_channels_list = [96, 192, 384, 768]
        return BackboneWithFPN(
            feat, return_layers=return_layers, in_channels_list=in_channels_list,
            out_channels=256, extra_blocks=LastLevelMaxPool(),
        )

    print('Custom ConvNeXt FPN skeleton defined — wire carefully, then FasterRCNN(backbone, num_classes=2)')
else:
    print('후순위 스켈레톤 비활성')